In [0]:
from pyspark.sql import functions as F

In [0]:
# country_vaccine
SOURCE_CATALOG_NAME_CV = 'covid_19'
SOURCE_SCHEMA_NAME_CV = 'silver'
SOURCE_TABLE_NAME_CV = 'country_vaccine'

# dim_country
SOURCE_CATALOG_NAME_DC = 'covid_19'
SOURCE_SCHEMA_NAME_DC = 'gold'
SOURCE_TABLE_NAME_DC = 'dim_country'

# dim_vaccine
SOURCE_CATALOG_NAME_DV = 'covid_19'
SOURCE_SCHEMA_NAME_DV = 'gold'
SOURCE_TABLE_NAME_DV = 'dim_vaccine'

# bridge_country_vaccine
TARGET_CATALOG_NAME = 'covid_19'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'bridge_country_vaccine'

In [0]:
df_country_vaccine = spark.table(f'{SOURCE_CATALOG_NAME_CV}.{SOURCE_SCHEMA_NAME_CV}.{SOURCE_TABLE_NAME_CV}')
df_dim_country = spark.table(f'{SOURCE_CATALOG_NAME_DC}.{SOURCE_SCHEMA_NAME_DC}.{SOURCE_TABLE_NAME_DC}')
df_dim_vaccine = spark.table(f'{SOURCE_CATALOG_NAME_DV}.{SOURCE_SCHEMA_NAME_DV}.{SOURCE_TABLE_NAME_DV}')

In [0]:
df_bridge_country_vaccine = (
    df_country_vaccine
    .join(
        df_dim_country,
        on=['country', 'iso_code'],
        how='inner'
    )
    .join(
        df_dim_vaccine,
        on='vaccine_name',
        how='inner'
    )
    .select(
        'country_key',
        'vaccine_key'
    )
    .dropDuplicates()
)

In [0]:
df_bridge_country_vaccine\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')